# Vigor Dynamics Along the Threat Imminence Continuum

Three questions:
1. **Pre-encounter:** Do people press harder when threat is higher? (within cookie type)
2. **Encounter:** What happens to pressing the moment the predator appears?
3. **Terminal:** What happens to pressing as the predator approaches strike?

**Metric:** Normalized press rate = median(1/IPI) / calibrationMax. Scale 0 to ~1.5. Same physical scale regardless of cookie type. Cookie type controlled by running analyses separately for heavy and light, and by including cookie as a covariate.

**Resolution:** 200ms bins from raw keypress timestamps (5Hz native resolution), 3-point moving average (600ms) for display.

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, ast
import matplotlib.pyplot as plt
from scipy.stats import ttest_rel, pearsonr, ttest_1samp
from pathlib import Path
import os

%matplotlib inline
plt.rcParams.update({'figure.dpi': 120, 'font.size': 10,
                     'axes.spines.right': False, 'axes.spines.top': False})

# Find repo root (walk up until we find CLAUDE.md or .git)
_nb_dir = Path(os.getcwd())
REPO_ROOT = _nb_dir
for _ in range(5):
    if (REPO_ROOT / '.git').exists() or (REPO_ROOT / 'CLAUDE.md').exists():
        break
    REPO_ROOT = REPO_ROOT.parent

DATA_DIR = REPO_ROOT / "data/exploratory_350/processed/stage5_filtered_data_20260320_191950"
RESULTS_DIR = REPO_ROOT / "results/stats"
EXCLUDE = [154, 197, 208]
BIN = 0.2  # 200ms
SMOOTH = 3  # 3-point moving average

print(f"Repo root: {REPO_ROOT}")
print(f"Data dir exists: {DATA_DIR.exists()}")

beh = pd.read_csv(DATA_DIR / "behavior_rich.csv", low_memory=False)
beh = beh[~beh['subj'].isin(EXCLUDE)].copy()
beh['T_round'] = beh['threat'].round(1)
beh['is_heavy'] = (beh['trialCookie_weight'] == 3.0).astype(int)
beh['actual_req'] = np.where(beh['is_heavy'] == 1, 0.9, 0.4)
beh['enc_t'] = pd.to_numeric(beh['encounterTime'], errors='coerce')
beh['strike_t'] = pd.to_numeric(beh['strike_time'], errors='coerce')
beh['is_attack'] = beh['isAttackTrial'].astype(int)

print(f"Loaded {len(beh)} trials, {beh['subj'].nunique()} subjects")

## Compute timecourse data from raw keypresses

For each trial, compute normalized press rate in 200ms bins across three timeframes:
- **Onset-aligned** (time from trial start): shows the ramp-up and pre-encounter threat modulation
- **Encounter-aligned** (time from encounterTime): shows the encounter response
- **Strike-aligned** (time from strike_time, attack trials only): shows terminal behavior

In [ ]:
%%time

WIN_ONSET = np.arange(0, 6.0, BIN)
WIN_ENC = np.arange(-2.0, 5.0, BIN)
WIN_STRIKE = np.arange(-5.0, 1.0, BIN)

onset_recs, enc_recs, strike_recs = [], [], []

for _, row in beh.iterrows():
    try:
        pt = np.array(ast.literal_eval(str(row['alignedEffortRate'])), dtype=float)
        if len(pt) < 5: continue
        cal = row['calibrationMax']; enc = row['enc_t']; strike = row['strike_t']
        if cal <= 0 or pd.isna(enc): continue

        ipis = np.diff(pt)
        midpoints = (pt[:-1] + pt[1:]) / 2
        rates = np.where(ipis > 0.01, (1.0 / ipis) / cal, np.nan)

        base = dict(subj=row['subj'], T_round=row['T_round'],
                    cookie=row['is_heavy'], is_attack=row['is_attack'])

        # Onset-aligned
        for t0 in WIN_ONSET:
            mask = (midpoints >= t0) & (midpoints < t0 + BIN)
            v = rates[mask]; v = v[~np.isnan(v)]
            if len(v) >= 1:
                onset_recs.append({**base, 't_bin': round(t0 + BIN/2, 2), 'rate': np.median(v)})

        # Encounter-aligned
        enc_mid = midpoints - enc
        for t0 in WIN_ENC:
            mask = (enc_mid >= t0) & (enc_mid < t0 + BIN)
            v = rates[mask]; v = v[~np.isnan(v)]
            if len(v) >= 1:
                enc_recs.append({**base, 't_bin': round(t0 + BIN/2, 2), 'rate': np.median(v)})

        # Strike-aligned (attack only)
        if row['is_attack'] == 1 and not pd.isna(strike) and strike > enc + 1:
            strike_mid = midpoints - strike
            for t0 in WIN_STRIKE:
                mask = (strike_mid >= t0) & (strike_mid < t0 + BIN)
                v = rates[mask]; v = v[~np.isnan(v)]
                if len(v) >= 1:
                    strike_recs.append({**base, 't_bin': round(t0 + BIN/2, 2), 'rate': np.median(v)})
    except: pass

onset_df = pd.DataFrame(onset_recs)
enc_df = pd.DataFrame(enc_recs)
strike_df = pd.DataFrame(strike_recs)

print(f"Onset: {len(onset_df):,} records")
print(f"Encounter: {len(enc_df):,} records")
print(f"Strike: {len(strike_df):,} records")

## 1. Pre-encounter: Threat modulates vigor within cookie type

Onset-aligned normalized press rate, separated by heavy and light cookies. Three lines per panel (T=0.1, 0.5, 0.9). If threat genuinely drives vigor (not just shifting choice toward light cookies), the lines should separate within each cookie type.

In [ ]:
def smooth(s, w=SMOOTH):
    return s.rolling(w, center=True, min_periods=1).mean()

# Per-subject first, then average
def agg_timecourse(df, group_cols, val_col='rate', min_count=100):
    subj = df.groupby(['subj'] + group_cols + ['t_bin'])[val_col].mean().reset_index()
    agg = subj.groupby(group_cols + ['t_bin'])[val_col].agg(['mean', 'sem', 'count']).reset_index()
    return agg[agg['count'] >= 30]

C = {0.1: '#2196F3', 0.5: '#9E9E9E', 0.9: '#F44336'}

fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

for ax, cookie, title in [(axes[0], 1, 'Heavy cookies (req=0.9)'),
                           (axes[1], 0, 'Light cookies (req=0.4)')]:
    sub = onset_df[onset_df['cookie'] == cookie]
    ts = agg_timecourse(sub, ['T_round'])

    for T in [0.1, 0.5, 0.9]:
        d = ts[ts['T_round'] == T].sort_values('t_bin')
        m = smooth(d['mean']); s = smooth(d['sem'])
        ax.plot(d['t_bin'], m, color=C[T], lw=2, label=f'T={T}')
        ax.fill_between(d['t_bin'], m - s, m + s, color=C[T], alpha=0.15)

    ax.set_xlabel('Time from trial start (s)')
    ax.set_title(title, fontweight='bold', fontsize=12)
    ax.legend(fontsize=9, frameon=False)

axes[0].set_ylabel('Normalized press rate\n(median IPI⁻¹ / calibMax)')
plt.suptitle('1. Pre-encounter: Threat modulates vigor within cookie type', fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# Stats: paired t-test T=0.9 vs T=0.1 for trial-level mean rate
print("\nPaired t-test (T=0.9 vs T=0.1, per-subject mean rate, full trial):")
vigor_metrics_path = RESULTS_DIR / "vigor_analysis/vigor_metrics.csv"
if vigor_metrics_path.exists():
    metrics = pd.read_csv(vigor_metrics_path)
    full = metrics[metrics['epoch'] == 'full']
    for ck, label in [(1, 'Heavy'), (0, 'Light')]:
        sub = full[full['cookie'] == ck]
        lo = sub[sub['T_round'] == 0.1].groupby('subj')['norm_rate'].mean()
        hi = sub[sub['T_round'] == 0.9].groupby('subj')['norm_rate'].mean()
        shared = sorted(set(lo.index) & set(hi.index))
        diff = hi.loc[shared].values - lo.loc[shared].values
        t, p = ttest_rel(hi.loc[shared], lo.loc[shared])
        d = diff.mean() / diff.std()
        print(f"  {label}: T=0.1={lo.loc[shared].mean():.4f}, T=0.9={hi.loc[shared].mean():.4f}, "
              f"Δ={diff.mean():+.4f}, t={t:.2f}, p={p:.2e}, d={d:+.3f}")
else:
    print("  (Run scripts/vigor_pipeline/compute_metrics.py first to generate vigor_metrics.csv)")

## 2. Encounter: The moment threat becomes real

Two plots:
- **Left:** Encounter-aligned press rate by threat level (heavy and light cookies separately). Shows whether the encounter response differs by threat.
- **Right:** Attack minus non-attack difference (per-subject, then averaged). Isolates the encounter effect — what the predator's appearance adds above baseline trial dynamics. The cleanest measure of the reactive motor response.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Panel A: Heavy cookies encounter-aligned by threat
ax = axes[0]
sub = enc_df[enc_df['cookie'] == 1]
ts = agg_timecourse(sub, ['T_round'])
for T in [0.1, 0.5, 0.9]:
    d = ts[ts['T_round'] == T].sort_values('t_bin')
    m = smooth(d['mean']); s = smooth(d['sem'])
    ax.plot(d['t_bin'], m, color=C[T], lw=2, label=f'T={T}')
    ax.fill_between(d['t_bin'], m - s, m + s, color=C[T], alpha=0.15)
ax.axvline(0, color='gray', ls='--', lw=1, alpha=0.5)
ax.set_xlabel('Time from encounter (s)')
ax.set_ylabel('Normalized press rate')
ax.set_title('A. Heavy cookies by threat', fontweight='bold')
ax.legend(fontsize=9, frameon=False)

# Panel B: Light cookies encounter-aligned by threat
ax = axes[1]
sub = enc_df[enc_df['cookie'] == 0]
ts = agg_timecourse(sub, ['T_round'])
for T in [0.1, 0.5, 0.9]:
    d = ts[ts['T_round'] == T].sort_values('t_bin')
    m = smooth(d['mean']); s = smooth(d['sem'])
    ax.plot(d['t_bin'], m, color=C[T], lw=2, label=f'T={T}')
    ax.fill_between(d['t_bin'], m - s, m + s, color=C[T], alpha=0.15)
ax.axvline(0, color='gray', ls='--', lw=1, alpha=0.5)
ax.set_xlabel('Time from encounter (s)')
ax.set_title('B. Light cookies by threat', fontweight='bold')
ax.legend(fontsize=9, frameon=False)

# Panel C: Attack - no-attack difference (per-subject, all cookies)
ax = axes[2]
subj_ts = enc_df.groupby(['subj', 'is_attack', 't_bin'])['rate'].mean().reset_index()
att = subj_ts[subj_ts['is_attack'] == 1].rename(columns={'rate': 'att'}).drop(columns='is_attack')
non = subj_ts[subj_ts['is_attack'] == 0].rename(columns={'rate': 'non'}).drop(columns='is_attack')
diff = att.merge(non, on=['subj', 't_bin'])
diff['effect'] = diff['att'] - diff['non']
panel_c = diff.groupby('t_bin')['effect'].agg(['mean', 'sem']).reset_index()

m = smooth(panel_c['mean']); s = smooth(panel_c['sem'])
ax.fill_between([0, 5], m.min() - 0.01, m.max() + 0.02, color='#FFEBEE', alpha=0.4, zorder=0)
ax.plot(panel_c['t_bin'], m, color='#F44336', lw=2)
ax.fill_between(panel_c['t_bin'], m - 1.96*s, m + 1.96*s, color='#F44336', alpha=0.2)
ax.axhline(0, color='gray', ls='-', lw=0.8, alpha=0.5)
ax.axvline(0, color='gray', ls='--', lw=1, alpha=0.5)
ax.set_xlabel('Time from encounter (s)')
ax.set_ylabel('Δ Press rate (attack − no attack)')
ax.set_title('C. Encounter effect (within-subject)', fontweight='bold')

peak = panel_c.loc[panel_c['mean'].idxmax()]
ax.annotate(f'Peak: +{peak["mean"]:.3f} at t={peak["t_bin"]:.1f}s',
            xy=(peak['t_bin'], peak['mean']), xytext=(peak['t_bin']+0.5, peak['mean']+0.005),
            fontsize=9, color='#F44336')

plt.suptitle('2. Encounter: The moment threat becomes real', fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# Stats from precomputed metrics if available
print(f"\nEncounter spike (attack - non-attack, normalized rate):")
if vigor_metrics_path.exists():
    sub_atk = metrics[metrics['epoch']=='reactive']
    s_a = sub_atk[sub_atk['is_attack']==1].groupby('subj')['norm_rate'].mean()
    s_n = sub_atk[sub_atk['is_attack']==0].groupby('subj')['norm_rate'].mean()
    shared = sorted(set(s_a.index) & set(s_n.index))
    spike = s_a.loc[shared] - s_n.loc[shared]
    t, p = ttest_1samp(spike, 0)
    d = spike.mean() / spike.std()
    print(f"  Mean spike: {spike.mean():+.4f}, t={t:.2f}, p={p:.2e}, d={d:+.3f}")
    print(f"  {(spike>0).mean()*100:.0f}% of subjects show positive spike")

    for T in [0.1, 0.5, 0.9]:
        sa = sub_atk[(sub_atk['is_attack']==1)&(sub_atk['T_round']==T)].groupby('subj')['norm_rate'].mean()
        sn = sub_atk[(sub_atk['is_attack']==0)&(sub_atk['T_round']==T)].groupby('subj')['norm_rate'].mean()
        sh = sorted(set(sa.index)&set(sn.index))
        sp = sa.loc[sh] - sn.loc[sh]
        print(f"  T={T}: spike={sp.mean():+.4f}")

## 3. Terminal: What happens as the predator approaches

Strike-aligned timecourse (attack trials only). t=0 is the moment the predator strikes. Negative time = before strike. Shows the last ~5 seconds of the trial as the predator closes in.

Key questions:
- Does pressing rate change as the predator approaches?
- Does threat level matter in terminal? (Prior finding: threat drops out of terminal epoch)
- Heavy vs light: different terminal dynamics?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

# Panel A: Strike-aligned by threat, heavy cookies
ax = axes[0]
sub = strike_df[strike_df['cookie'] == 1]
ts = agg_timecourse(sub, ['T_round'])
for T in [0.1, 0.5, 0.9]:
    d = ts[ts['T_round'] == T].sort_values('t_bin')
    if len(d) < 3: continue
    m = smooth(d['mean']); s = smooth(d['sem'])
    ax.plot(d['t_bin'], m, color=C[T], lw=2, label=f'T={T}')
    ax.fill_between(d['t_bin'], m - s, m + s, color=C[T], alpha=0.15)
ax.axvline(0, color='red', ls='-', lw=2, alpha=0.5, label='Strike')
ax.set_xlabel('Time from predator strike (s)')
ax.set_ylabel('Normalized press rate')
ax.set_title('A. Heavy cookies — terminal', fontweight='bold')
ax.legend(fontsize=9, frameon=False)

# Panel B: Strike-aligned by threat, light cookies
ax = axes[1]
sub = strike_df[strike_df['cookie'] == 0]
ts = agg_timecourse(sub, ['T_round'])
for T in [0.1, 0.5, 0.9]:
    d = ts[ts['T_round'] == T].sort_values('t_bin')
    if len(d) < 3: continue
    m = smooth(d['mean']); s = smooth(d['sem'])
    ax.plot(d['t_bin'], m, color=C[T], lw=2, label=f'T={T}')
    ax.fill_between(d['t_bin'], m - s, m + s, color=C[T], alpha=0.15)
ax.axvline(0, color='red', ls='-', lw=2, alpha=0.5, label='Strike')
ax.set_xlabel('Time from predator strike (s)')
ax.set_title('B. Light cookies — terminal', fontweight='bold')
ax.legend(fontsize=9, frameon=False)

plt.suptitle('3. Terminal: Pressing as predator approaches strike', fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# Stats: terminal epoch threat modulation
print("\nTerminal vigor by threat (within cookie type, paired t-test):")
term = metrics[metrics['epoch'] == 'terminal']
for ck, label in [(1, 'Heavy'), (0, 'Light')]:
    sub = term[term['cookie'] == ck]
    lo = sub[sub['T_round'] == 0.1].groupby('subj')['norm_rate'].mean()
    hi = sub[sub['T_round'] == 0.9].groupby('subj')['norm_rate'].mean()
    shared = sorted(set(lo.index) & set(hi.index))
    if len(shared) > 20:
        diff = hi.loc[shared].values - lo.loc[shared].values
        t, p = ttest_rel(hi.loc[shared], lo.loc[shared])
        d = diff.mean() / diff.std()
        print(f"  {label}: T=0.1={lo.loc[shared].mean():.4f}, T=0.9={hi.loc[shared].mean():.4f}, "
              f"Δ={diff.mean():+.4f}, t={t:.2f}, p={p:.2e}, d={d:+.3f}")

## 4. Individual differences: Who shows these effects?

Connect the vigor dynamics to the 2+2 model parameters (ce, cd). 
- Does cd predict the encounter spike?
- Does ce predict trial-level vigor?
- Frac_full by condition as the mechanistically relevant metric (determines actual survival)

In [ ]:
# Load 2+2 params
params_path = RESULTS_DIR / "oc_evc_final_params.csv"
params_22 = pd.read_csv(params_path)
params_22 = params_22[~params_22['subj'].isin(EXCLUDE)]
params_22['log_ce'] = np.log(params_22['c_effort'])
params_22['log_cd'] = np.log(params_22['c_death'])

# Encounter spike per subject (norm_rate, attack - non-attack in reactive epoch)
if vigor_metrics_path.exists():
    sub_atk = metrics[(metrics['epoch']=='reactive')&(metrics['is_attack']==1)].groupby('subj')['norm_rate'].mean()
    sub_noatk = metrics[(metrics['epoch']=='reactive')&(metrics['is_attack']==0)].groupby('subj')['norm_rate'].mean()
    shared = sorted(set(sub_atk.index) & set(sub_noatk.index))
    spike = pd.DataFrame({'subj': shared, 'encounter_spike': (sub_atk.loc[shared] - sub_noatk.loc[shared]).values})

    # Mean frac_full per subject
    subj_ff = metrics[metrics['epoch']=='full'].groupby('subj')['frac_full'].mean().reset_index()

    # Merge
    merged = spike.merge(params_22, on='subj').merge(subj_ff, on='subj')

    fig, axes = plt.subplots(1, 3, figsize=(16, 5))

    # A: cd → encounter spike
    ax = axes[0]
    r, p = pearsonr(merged['log_cd'], merged['encounter_spike'])
    ax.scatter(merged['log_cd'], merged['encounter_spike'], s=15, alpha=0.4, color='#F44336')
    m, b = np.polyfit(merged['log_cd'], merged['encounter_spike'], 1)
    x = np.linspace(merged['log_cd'].min(), merged['log_cd'].max(), 100)
    ax.plot(x, m*x+b, 'k-', lw=2)
    ax.set_xlabel('log(cd) — capture aversion')
    ax.set_ylabel('Encounter spike\n(attack − no attack press rate)')
    ax.set_title(f'A. cd → encounter spike (r={r:.3f})', fontweight='bold')
    ax.axhline(0, color='gray', ls='--', lw=0.5)

    # B: cd → frac_full
    ax = axes[1]
    r2, p2 = pearsonr(merged['log_cd'], merged['frac_full'])
    ax.scatter(merged['log_cd'], merged['frac_full'], s=15, alpha=0.4, color='#4CAF50')
    m, b = np.polyfit(merged['log_cd'], merged['frac_full'], 1)
    ax.plot(x, m*x+b, 'k-', lw=2)
    ax.set_xlabel('log(cd) — capture aversion')
    ax.set_ylabel('Fraction at full speed')
    ax.set_title(f'B. cd → frac_full (r={r2:.3f})', fontweight='bold')

    # C: ce → frac_full (expect null)
    ax = axes[2]
    r3, p3 = pearsonr(merged['log_ce'], merged['frac_full'])
    ax.scatter(merged['log_ce'], merged['frac_full'], s=15, alpha=0.4, color='#2196F3')
    m, b = np.polyfit(merged['log_ce'], merged['frac_full'], 1)
    x2 = np.linspace(merged['log_ce'].min(), merged['log_ce'].max(), 100)
    ax.plot(x2, m*x2+b, 'k-', lw=2)
    ax.set_xlabel('log(ce) — effort avoidance')
    ax.set_ylabel('Fraction at full speed')
    ax.set_title(f'C. ce → frac_full (r={r3:.3f})', fontweight='bold')

    plt.suptitle('4. Individual differences: ce (avoidance) vs cd (activation)', fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()

    print(f"\nParameter → vigor correlations:")
    print(f"  cd → encounter spike: r={r:+.3f}, p={p:.4f}")
    print(f"  cd → frac_full:       r={r2:+.3f}, p={p2:.4f}")
    print(f"  ce → encounter spike: r={pearsonr(merged['log_ce'], merged['encounter_spike'])[0]:+.3f}")
    print(f"  ce → frac_full:       r={r3:+.3f}, p={p3:.4f}")
else:
    print("Run vigor pipeline first (scripts/vigor_pipeline/compute_metrics.py)")

## 5. Summary: The full timecourse in one figure

Three-panel figure showing the full trial timeline:
- **Onset to encounter:** threat drives vigor ramp (within cookie type)
- **Encounter:** predator appearance triggers a motor spike (threat-independent)
- **Terminal:** pressing approaches strike (distance-driven, threat drops out)

In [ ]:
# Combined figure: onset → encounter → strike for HEAVY cookies only (cleanest signal)
# Using normalized press rate, per-subject averages, 600ms smoothing

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Panel 1: Onset-aligned (0 to 4s)
ax = axes[0]
sub = onset_df[onset_df['cookie'] == 1]
ts = agg_timecourse(sub, ['T_round'])
ts = ts[ts['t_bin'] <= 4.0]
for T in [0.1, 0.5, 0.9]:
    d = ts[ts['T_round'] == T].sort_values('t_bin')
    m = smooth(d['mean']); s = smooth(d['sem'])
    ax.plot(d['t_bin'], m, color=C[T], lw=2, label=f'T={T}')
    ax.fill_between(d['t_bin'], m - s, m + s, color=C[T], alpha=0.15)
ax.set_xlabel('Time from trial start (s)')
ax.set_ylabel('Normalized press rate')
ax.set_title('Pre-encounter\n(onset-aligned)', fontweight='bold', fontsize=13)
ax.legend(fontsize=9, frameon=False)
ax.annotate('Threat drives\nanticipatory vigor', xy=(2.5, 0.95), fontsize=9,
            color='gray', ha='center', style='italic')

# Panel 2: Encounter-aligned (-1.5 to 3.5s), heavy only
ax = axes[1]
sub = enc_df[enc_df['cookie'] == 1]
ts = agg_timecourse(sub, ['T_round'])
ts = ts[(ts['t_bin'] >= -1.5) & (ts['t_bin'] <= 3.5)]
for T in [0.1, 0.5, 0.9]:
    d = ts[ts['T_round'] == T].sort_values('t_bin')
    m = smooth(d['mean']); s = smooth(d['sem'])
    ax.plot(d['t_bin'], m, color=C[T], lw=2, label=f'T={T}')
    ax.fill_between(d['t_bin'], m - s, m + s, color=C[T], alpha=0.15)
ax.axvline(0, color='red', ls='--', lw=2, alpha=0.6)
ax.set_xlabel('Time from encounter (s)')
ax.set_title('Encounter\n(predator appears)', fontweight='bold', fontsize=13)
ax.legend(fontsize=9, frameon=False)
ax.annotate('Predator\nappears', xy=(0.1, ax.get_ylim()[0] + 0.02), fontsize=9,
            color='red', alpha=0.7)

# Panel 3: Strike-aligned (-4 to 0s), heavy only
ax = axes[2]
sub = strike_df[strike_df['cookie'] == 1]
ts = agg_timecourse(sub, ['T_round'])
ts = ts[(ts['t_bin'] >= -4) & (ts['t_bin'] <= 0)]
for T in [0.1, 0.5, 0.9]:
    d = ts[ts['T_round'] == T].sort_values('t_bin')
    if len(d) < 3: continue
    m = smooth(d['mean']); s = smooth(d['sem'])
    ax.plot(d['t_bin'], m, color=C[T], lw=2, label=f'T={T}')
    ax.fill_between(d['t_bin'], m - s, m + s, color=C[T], alpha=0.15)
ax.axvline(0, color='darkred', ls='-', lw=2, alpha=0.6)
ax.set_xlabel('Time from predator strike (s)')
ax.set_title('Terminal\n(predator strikes)', fontweight='bold', fontsize=13)
ax.legend(fontsize=9, frameon=False)
ax.annotate('Strike', xy=(0, ax.get_ylim()[1] - 0.02), fontsize=9,
            color='darkred', ha='right', alpha=0.7)

plt.suptitle('Heavy cookies: Vigor across the threat imminence continuum',
             fontweight='bold', fontsize=14, y=1.04)
plt.tight_layout()
plt.show()

print("Three phases of the trial:")
print("  1. Pre-encounter: Higher threat → higher press rate (within heavy cookies)")
print("  2. Encounter: Press rate shifts at predator appearance (threat lines may converge)")
print("  3. Terminal: Final pressing as predator closes in")